In [27]:
import pandas as pd

# Original data with mismatched lengths
data = {'Name': ['Alice','Aman', 'Bob', 'Aman','Aman'], 'Age': [25, 30, 29, 33, 45, 50]}

# Finding the maximum length of the columns
max_len = max(len(data['Name']), len(data['Age']))

# Determine the column to pad
x = 'Age' if max_len == len(data['Name']) else 'Name'

# Padding the selected column with None
data[x] = data[x] + [None] * (max_len - len(data[x]))

# Creating DataFrame
df = pd.DataFrame(data)

# Adding a new column 'Salary'
S = [50000, 60000, 70000,100,10,100,1000,1000,1000]

# Handle padding based on the length of 'Salary'
if len(S) < max_len:
    df['Salary'] = S + [None] * (max_len - len(S))
elif len(S) > max_len:
    
    # Pad 'Name' and 'Age' columns if 'Salary' is longer
    df = pd.DataFrame({
        'Name': df['Name'].tolist() + [None] * (len(S) - max_len),
        'Age': df['Age'].tolist() + [None] * (len(S) - max_len),
        'Salary': S
    })
    max_len = len(S)  # Update max_len to the new length
 
# Display the DataFrame
print(df)

# Descriptive statistics
print(df.describe())


    Name   Age  Salary
0  Alice  25.0   50000
1   Aman  30.0   60000
2    Bob  29.0   70000
3   Aman  33.0     100
4   Aman  45.0      10
5   None  50.0     100
6   None   NaN    1000
7   None   NaN    1000
8   None   NaN    1000
             Age        Salary
count   6.000000      9.000000
mean   35.333333  20356.666667
std     9.892758  30152.684789
min    25.000000     10.000000
25%    29.250000    100.000000
50%    31.500000   1000.000000
75%    42.000000  50000.000000
max    50.000000  70000.000000


In [34]:
df1 = pd.DataFrame({'key': ['A', 'B', 'C'], 'value': [1, 2, 3]})
df2 = pd.DataFrame({'key': ['A', 'B', 'D'], 'value': [4, 5, 6]})
merged_df = pd.merge(df1, df2, on='key')
print(merged_df)

# grouped_df = df.groupby('Name')['Salary'].sum()

# Group by 'Name' and then by 'Age', sum by 'Salary'
grouped_df=df.groupby(['Name', 'Age'])['Salary'].sum().reset_index()

# Group by 'Name' and sum 'Age' and 'Salary'
# grouped_sum = df.groupby('Name')[['Age', 'Salary']].sum().reset_index()

print(grouped_df)



  key  value_x  value_y
0   A        1        4
1   B        2        5
    Name   Age  Salary
0  Alice  25.0   50000
1   Aman  30.0   60000
2   Aman  33.0     100
3   Aman  45.0      10
4    Bob  29.0   70000


## Join Need Indexing

### Join: Syntax is df1.join(df2, how='join_type'). The default type of join is left, which means it includes all records from the left DataFrame and the matched records from the right DataFrame.

In [37]:
import pandas as pd

# Original data
data = {
    'Name': ['Alice', 'Bob', 'Aman', 'Alice', 'Bob', 'Aman'],
    'Age': [25, 30, 29, 25, 30, 29],
    'Salary': [50000, 60000, 70000, 80000, 90000, 100000],
    'id': [1, 2, 3, 1, 2, 3]
}

df = pd.DataFrame(data)

# New DataFrame df_info
data_info = {
    'id': [1, 2, 3],
    'company': ['Company A', 'Company B', 'Company C'],
    'location': ['Location A', 'Location B', 'Location C']
}

df_info = pd.DataFrame(data_info)

# Set 'id' as the index for df_info before joining
df_info.set_index('id', inplace=True)

# Join df with df_info based on 'id'
joined_df = df.join(df_info, on='id')

print("Joined DataFrame:")
print(joined_df)


Joined DataFrame:
    Name  Age  Salary  id    company    location
0  Alice   25   50000   1  Company A  Location A
1    Bob   30   60000   2  Company B  Location B
2   Aman   29   70000   3  Company C  Location C
3  Alice   25   80000   1  Company A  Location A
4    Bob   30   90000   2  Company B  Location B
5   Aman   29  100000   3  Company C  Location C


## Merging 

### Merge: Syntax is typically pd.merge(df1, df2, how='join_type', suffixes=('_left', '_right'),on='column_name'). The default type of join is inner, which means it returns only the intersection of the keys.

In [36]:
# Merging df with df_info on 'id'
merged_df = pd.merge(df, df_info, on='id')

print("Merged DataFrame:")
print(merged_df)

Merged DataFrame:
    Name  Age  Salary  id    company    location
0  Alice   25   50000   1  Company A  Location A
1    Bob   30   60000   2  Company B  Location B
2   Aman   29   70000   3  Company C  Location C
3  Alice   25   80000   1  Company A  Location A
4    Bob   30   90000   2  Company B  Location B
5   Aman   29  100000   3  Company C  Location C


### 1. **Optimizing the Join Between a Huge Transaction Table and a Master Table:**

When you have a large transaction table (with daily records) that you need to join with a master table, the goal is to make this join operation as fast as possible. Here’s how you can do that:

- **Indexing:** Think of an index as a shortcut to quickly find the data you need. By creating an index on the columns you’re joining on, the database can quickly locate the matching rows, speeding up the join.
  
- **Filtering Data First:** If you don’t need all the rows in the transaction table, filter out the unnecessary data before joining. This reduces the amount of data the database has to work with, making the join faster.

- **Partitioning:** If your transaction table is very large, you can split it into smaller pieces based on date or some other logical category. The database can then join only the relevant piece instead of the whole table.

- **Using the Right Join Type:** Depending on what data you need, choose the right kind of join (e.g., INNER JOIN for matching records only, LEFT JOIN to keep all records from one table). This ensures the database doesn’t do more work than necessary.

### 2. **Optimizing When You Need to Execute the Join in 15 Queries:**

If you need to break the join into 15 separate queries, here’s how you can make sure it’s done efficiently:

- **Divide the Work:** Break the data into 15 smaller, manageable chunks. For example, if your data is based on dates, each query could handle a specific date range. This way, each query processes less data, making it faster.

- **Run Queries at the Same Time:** If your system allows, run the 15 queries simultaneously instead of one after the other. This reduces the total time it takes to complete all the queries.

- **Reuse Results:** If some parts of the data are used in multiple queries, store those results temporarily so you don’t have to process the same data multiple times.

- **Combine Results Efficiently:** After running the 15 queries, combine their results in the most straightforward way possible. If the queries can be merged using a simple operation like UNION, do that to avoid unnecessary complexity.

- **Avoid Overlap:** Ensure each query only processes the data it needs. This prevents the system from doing redundant work.

By following these strategies, you can handle large datasets efficiently, even when you need to split the work across multiple queries.

## Multi Nested Dic,

In [41]:
nested_dict = {
    'key1': {'subkey1': {'subsubkey1': 'value1'}, 'subkey1b': {'subsubkey1b': 'value1b'}},
    'key2': {'subkey2': {'subsubkey2': 'value2'}, 'subkey2b': {'subsubkey2b': 'Aman'}}
}

def get_nested_value(data, target_key):
    if isinstance(data, dict):
        for key, value in data.items():
            if key == target_key:
                return value
            elif isinstance(value, dict):
                # Recursively search this dictionary
                result = get_nested_value(value, target_key)
                if result is not None:
                    return result
    return None

# print(get_nested_value(nested_dict, ['key1', 'subkey1', 'subsubkey1']))  # Output: 'value1'

# Testing the function with the target key 'subsubkey2b'
result = get_nested_value(nested_dict, 'subsubkey2b')
print(result)  # Expected output: 'Aman'


Aman


## List of Strings - Frequency of Each Element

In [43]:
from collections import Counter

my_list = ['apple', 'banana', 'apple', 'orange', 'banana', 'apple']
freq_count = Counter(my_list)
print(freq_count)  # Output: Counter({'apple': 3, 'banana': 2, 'orange': 1})

freq_list = list(freq_count.items())  # As list
print(freq_list)  # Output: [('apple', 3), ('banana', 2), ('orange', 1)]

freq_str = ', '.join(f"{key}: {value}" for key, value in freq_count.items())  # As string
print(freq_str)  # Output: 'apple: 3, banana: 2, orange: 1'

s=my_list.count('apple')
print(s)

Counter({'apple': 3, 'banana': 2, 'orange': 1})
[('apple', 3), ('banana', 2), ('orange', 1)]
apple: 3, banana: 2, orange: 1
3


## Input List to Output String

In [44]:
input_list = ['This', 'is', 'a', 'test']
output_string = ' '.join(input_list)
print(output_string)  # Output: 'This is a test'


This is a test


## count the occurrences of a specific input string in a file`

In [52]:
def count_occurrences(file_path, search_string):
    try:
        with open(file_path, 'r') as file:
            content = file.read()
            occurrences = content.count(search_string)
            return occurrences
    except FileNotFoundError:
        print(f"The file {file_path} does not exist.")
        return 0

# Example usage
file_path = 'example.txt'
search_string = 'Pending'

occurrences = count_occurrences(file_path, search_string)
print(f"The string '{search_string}' occurs {occurrences} times in the file.")


The string 'Pending' occurs 5 times in the file.


### Without Pre Defined Functions

In [54]:
def count_occurrences(file_path, search_string):
    try:
        with open(file_path, 'r') as file:
            content = file.read()
            
            search_len = len(search_string)
            count = 0
            i = 0
            
            while i <= len(content) - search_len:
                # Check if the substring matches the search string
                if content[i:i + search_len] == search_string:
                    count += 1
                    i += search_len  # Move past this occurrence
                else:
                    i += 1  # Move to the next character
                    
            return count
    except FileNotFoundError:
        print(f"The file {file_path} does not exist.")
        return 0

# Example usage
file_path = 'example.txt'
search_string = 'Pending'

occurrences = count_occurrences(file_path, search_string)
print(f"The string '{search_string}' occurs {occurrences} times in the file.")


The string 'Pending' occurs 5 times in the file.


### 1. **Reversing a String Using a Predefined Function**

You can use Python's slicing feature, which is a predefined way to reverse a string:



**Explanation:**
- The slicing `s[::-1]` creates a new string that starts from the end and goes to the beginning, effectively reversing the string.



In [56]:

def reverse_string(s):
    return s[::-2]

# Example usage
input_string = "Hello, World!"
reversed_string = reverse_string(input_string)
print(f"Reversed string: {reversed_string}")


Reversed string: !dlroW ,olleH


### 2. **Reversing a String Without Using a Predefined Function**





**Explanation:**
- The program iterates over each character in the original string.
- For each character, it adds it to the beginning of a new string (`reversed_str`), effectively building the reversed string step by step.

Both methods will give you the reversed string, but the second method avoids using any of Python's built-in string manipulation features like slicing.

In [55]:

def reverse_string(s):
    reversed_str = ''
    for char in s:
        reversed_str = char + reversed_str
    return reversed_str

# Example usage
input_string = "Hello, World!"
reversed_string = reverse_string(input_string)
print(f"Reversed string: {reversed_string}")


Reversed string: !dlroW ,olleH


## Dynamic Programming: Divide a List into N Segments

In [48]:
def divide_list(lst, n):
    avg = len(lst) / n
    out = []
    last = 0.0

    while last < len(lst):
        out.append(lst[int(last):int(last + avg)])
        last += avg

    return out

my_list = [1, 2, 3, 4, 5, 6, 7, 8, 9,10,11]
segments = 3
result = divide_list(my_list, segments)
print(result)  # Output: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]


[[1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
